# Observational CSV workflow

Research question: how do we run a reproducible benchmark when the series comes from a study CSV and no ground truth is available?

Observational mode avoids truth-based accuracy claims and focuses on validity, stability, uncertainty width, and failure patterns.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from lrdbench.output_contract import validate_output_contract
from lrdbench.runner import run_manifest_mapping

In [ ]:
rng = np.random.default_rng(20260427)
data_dir = Path("workshop_data")
data_dir.mkdir(exist_ok=True)
t = np.arange(256)
values = 0.002 * t + 0.3 * np.sin(t / 13.0) + rng.normal(0.0, 0.05, size=t.size)
csv_path = data_dir / "participant_001_segment_a.csv"
pd.DataFrame({"value": values}).to_csv(csv_path, index=False)
csv_path

In [ ]:
manifest = {
    "manifest_id": "workshop_observational_csv_v1",
    "name": "workshop_observational_csv",
    "mode": "observational",
    "source": {
        "type": "csv_series_index",
        "series": [{"record_id": "participant_001_segment_a", "path": str(csv_path), "value_column": "value"}],
    },
    "estimators": [
        {
            "name": "RS",
            "family": "time_domain",
            "target_estimand": "hurst_scaling_proxy",
            "supports_ci": True,
            "params": {"n_bootstrap": 32, "bootstrap_block_len": 12, "ci_levels": [0.95]},
        }
    ],
    "metrics": ["validity_rate", "runtime", {"name": "ci_width", "levels": [0.95]}, "instability", "preprocessing_sensitivity"],
    "preprocessing": {"sensitivity_eps": 1.0e-4},
    "leaderboards": [
        {
            "name": "observational_robustness",
            "mode": "observational",
            "component_metrics": ["instability", "validity_rate", "runtime"],
            "weights": {"instability": 0.5, "validity_rate": 0.3, "runtime": 0.2},
            "ranking_rule": "weighted_rank",
            "tie_break_rule": "best_primary_metric",
        }
    ],
    "report": {"formats": ["html", "csv"], "export_root": "reports/notebooks/observational_csv"},
    "seeds": {"global_seed": 20260427},
    "execution": {"max_workers": 1},
}

In [ ]:
out = run_manifest_mapping(manifest, base_dir=Path.cwd())
errors = validate_output_contract(out.result_store_path)
assert errors == []
out.result_store_path

In [ ]:
result_store = Path(out.result_store_path)
summary = pd.read_csv(result_store / "tables" / "run_summary.csv")
metrics = pd.read_csv(result_store / "tables" / "per_stratum_metrics.csv")
summary[["run_id", "mode"]], metrics[["metric_name", "value"]].head()